# About this project

## Abstract
This project analysis the geographical distribution of tourism establishments in South Tyrol, Italy.

Through visualisations, the analysis aims at identifying parts of the region with the highest density of tourism establishments across various dimensions, such as:

- Number of rooms
- Occupancy levels (in absolute terms and per 1,000 inhabitants)
- Average occupancy level by establishment
- Number of establishments (in absolute terms and per 1,000 inhabitants)

The analysis is carried out both at the granularity of GPS coordinates as well as at the municipal level.
    
## Data Sources

This project uses the following data sources

- **Tourism** data provided by the [Opendatahub API.](https://tourism.opendatahub.bz.it/swagger/index.html#/Accommodation/SingleAccommodationRoom)

- **Population** and **municipal boundaries** provided by the [Geocatalogue of South Tyrol.](https://geonetwork1.civis.bz.it/geonetwork)
    - See the `README` at the root of this repo for detailed information on which data layers are used. 

# Analysis

## Environment

Preparing the environment

- Loading necessary `Python` libaries as well as custom functions

In [ ]:
%load_ext autoreload

In [ ]:
%autoreload 2

In [ ]:
from south_tyrol_tourism.config import settings, VARIABLES_INV
from south_tyrol_tourism.dashboard_data import load_municipality_data, load_density_data
from south_tyrol_tourism.visualisation import define_municipality_map, define_density_map

In [ ]:
from IPython.display import Image
import seaborn as sns
import matplotlib.pylab as plt
import matplotlib as mpl
import pandas as pd
import geoviews as gv
gv.extension('bokeh')
mpl.rcParams['figure.dpi'] = 300

In [ ]:
# Hide depreciation warnings, e.g. ShapelyDeprecationWarning
import warnings
warnings.filterwarnings("ignore")

## Data

This section prepares the data before visualising it.

In [ ]:
df_municipality = load_municipality_data()
df_municipality.head(2)

In [ ]:
d_establishments = load_density_data()
d_establishments.keys()

In [ ]:
df_tourism = pd.read_parquet(settings.prepared_accommodation_file)
df_tourism.head(2)

## EDA

This section performs Exploratory Data Anlysis (EDA) on various properties of tourism establishments

### Categories

In [ ]:
df_categories = (
    df_tourism
    .groupby(["AccoCategoryRating", "AccoCategoryType"])
    .agg(Count=("Id", "count"))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# Left: Category Type
type_counts = (
    df_categories.groupby("AccoCategoryType")["Count"].sum()
    .sort_values(ascending=True)
)
bars = axes[0].barh(type_counts.index, type_counts.values, color="#4C72B0")
x_max = type_counts.max()
for bar, val in zip(bars, type_counts.values):
    axes[0].text(
        bar.get_width() + x_max * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}", va="center", fontsize=9,
    )
axes[0].set_title("Category Types", fontsize=11)
axes[0].set_xlabel("Number of Establishments")
axes[0].spines[["top", "right"]].set_visible(False)
axes[0].set_xlim(right=x_max * 1.18)

# Right: Category Rating
rating_counts = (
    df_categories.groupby("AccoCategoryRating")["Count"].sum()
    .sort_values(ascending=True)
)
bars = axes[1].barh(rating_counts.index, rating_counts.values, color="#4C72B0")
x_max = rating_counts.max()
for bar, val in zip(bars, rating_counts.values):
    axes[1].text(
        bar.get_width() + x_max * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}", va="center", fontsize=9,
    )
axes[1].set_title("Category Rating", fontsize=11)
axes[1].set_xlabel("Number of Establishments")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].set_xlim(right=x_max * 1.18)

plt.tight_layout()

### Altitude vs Category type

In [ ]:
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(6, 3))
sns.despine(fig)
sns.histplot(
    df_tourism.query("Altitude > 0"),
    x="Altitude",
    hue="AccoCategoryType",
    multiple="stack",
    palette="muted",
)
ax.set_xlim(100, 3_000)
ax.set_ylabel("Number of Establishments")
ax.set_xlabel("Altitude [m]")

## Visualisations

This section creats the `geoviews` plots.

### At the level of municipalities

This analysis summarises tourism information **at the municipal level** by means of aggregation.

Specifically, the following metrics are available:

- Number of tourism establishments
- Number of tourism establishments per 1,000 inhabitants
- Total occupancy
- Total occupancy per 1,000 inhabitants
- Total number of rooms
- Mean occupancy per tourism establishment
- Share of
    - establishments by rating (1, 2, ..., 5)
    - establishments by category (flowers, stars, suns)

In [ ]:
kpi_to_visualise = "Number of Tourism Establishments"

In [ ]:
fig = define_municipality_map(
    data=df_municipality,
    color_col=VARIABLES_INV[kpi_to_visualise],
    title=kpi_to_visualise,
    clabel=kpi_to_visualise,
)
fig

In [ ]:
# For Github: visualise a static version of the above geoviews figure
filename = "../plots/municipality"
renderer = gv.renderer('matplotlib')
# renderer.save(fig, filename)
Image(filename=filename + ".png")

### At the level of individual establishments

This analysis performs a kernel density estimation over GPS coordinates of individual tourism to highlight regions in South Tyrol with the highest conglomeration of tourism establishments.

In [ ]:
fig = define_density_map(**d_establishments)
fig

In [ ]:
# For Github: visualise a static version of the above geoviews figure
filename = "../plots/spatial_density"
renderer = gv.renderer('matplotlib')
# renderer.save(fig, filename)
Image(filename=filename + ".png")